In [2]:
!pip install -q chromadb sentence-transformers pypdf langchain-text-splitters

In [3]:
from google.colab import files
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

In [4]:
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded File:", pdf_file)

Saving Strategic Implementation Plan for an MCP.pdf to Strategic Implementation Plan for an MCP.pdf
Uploaded File: Strategic Implementation Plan for an MCP.pdf


In [5]:
reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    if page.extract_text():
        text += page.extract_text()

print("Total Characters:", len(text))
print("\nFirst 500 Characters:\n")
print(text[:500])

Total Characters: 1793

First 500 Characters:

Strategic Implementation Plan for an MCP-Powered Enterprise Agent System (Summary) 
To solve OrionOps' scalability, reliability, and governance challenges, the proposed solution is a 
modular MCP-powered agent platform with secure tool integration, intelligent workflows, and 
strong monitoring. 
• Standardized MCP Architecture: Use MCP servers with stdio (local) and HTTP (distributed) 
transports. Organize tools using namespaces (e.g., documents.*, operations.*) and expose 
enterprise data as se


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(text)

print("Number of Chunks:", len(chunks))
print("\nFirst Chunk:\n")
print(chunks[0])

Number of Chunks: 5

First Chunk:

Strategic Implementation Plan for an MCP-Powered Enterprise Agent System (Summary) 
To solve OrionOps' scalability, reliability, and governance challenges, the proposed solution is a 
modular MCP-powered agent platform with secure tool integration, intelligent workflows, and 
strong monitoring. 
• Standardized MCP Architecture: Use MCP servers with stdio (local) and HTTP (distributed) 
transports. Organize tools using namespaces (e.g., documents.*, operations.*) and expose


In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [8]:
client = chromadb.Client()

collection = client.create_collection(name="pdf_collection")

print("ChromaDB Collection Created")

ChromaDB Collection Created


In [9]:
embeddings = model.encode(chunks).tolist()

ids = [str(i) for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=embeddings
)

print("Stored", len(chunks), "chunks successfully!")

Stored 5 chunks successfully!


In [ ]:
while True:

    query = input("\nAsk a question (type 'exit' to stop): ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )

    print("\nTop 3 Relevant Snippets:\n")

    for i, doc in enumerate(results["documents"][0], start=1):
        print(f"Snippet {i}")
        print(doc)
        print("-"*70)


Ask a question (type 'exit' to stop): what is the title?

Top 3 Relevant Snippets:

Snippet 1
• Reflexive Reasoning and Self-Healing: Implement an actor–critic model where one agent 
generates responses and another reviews them for accuracy. Add retries, validation, fallback 
tools, and human escalation to improve reliability and safely handle failures. 
• Structured Multi-Step Workflows: Use a Planner → Executor → Validator pipeline to break 
complex tasks into smaller steps, execute them, and validate results at each stage, reducing 
errors and improving output quality.
----------------------------------------------------------------------
Snippet 2
transports. Organize tools using namespaces (e.g., documents.*, operations.*) and expose 
enterprise data as secure, read-only URI-based MCP resources. 
• Smart Discovery and Routing: Agents automatically discover available MCP servers and their 
capabilities. Requests are routed dynamically to the most suitable tools or servers based on